In [ ]:
#| default_exp features.endmotif

In [ ]:
#| export
from __future__ import annotations
import pandas as pd
import structlog
from kreview.eval_engine import FeatureEvaluator

log = structlog.get_logger()
        

In [ ]:
#| export
def _parse_array(s):
    if not isinstance(s, str) or not s.startswith('['): 
        return []
    clean = s.replace('[', '').replace(']', '').replace(chr(10), '').replace(chr(13), '')
    try:
        return [float(x) for x in clean.split()]
    except:
        return []

class EndMotifEvaluator(FeatureEvaluator):
    """Extracts 4-mer fragment end frequencies."""
    
    name = "EndMotif"
    source_file = ".EndMotif.ontarget.parquet"
    tier = 2
    category = "epigenetics_and_geometry"

    def extract(self, df: pd.DataFrame) -> dict[str, float]:
        extracted = {}
        try:
            if df.empty:
                return extracted
            cols = set(df.columns)

            self.tier = 3
            self.category = "motifs"
            if "Motif" in cols and "Frequency" in cols:
                for row in df.to_dict('records'):
                    m = str(row["Motif"])
                    if pd.notna(row["Frequency"]):
                        extracted[f"endmotif_{m}"] = float(row["Frequency"])
    
            return extracted
        except Exception as e:
            log.warning("endmotif_extraction_failed", error=str(e))
            return {}


In [ ]:
#| test
evaluator = EndMotifEvaluator()
df = pd.DataFrame([{"Motif": "AAAA", "Frequency": 0.05}])
res = evaluator.extract(df)
assert res["endmotif_AAAA"] == 0.05
